In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install numpy==2.1.3

In [ ]:
!pip install pandas==2.2.3

In [ ]:
!pip install matplotlib==3.9.2

In [ ]:
!pip install tabulate==0.9.0

In [ ]:
!pip freeze

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

In [ ]:
project_id: int = 0 # @TODO: Set a project ID.

In [ ]:
conn = sqlite3.connect('../database/db.sqlite')

Number of test in-/exclusions

In [ ]:
query = f"SELECT project_id, count(*) AS total_count, sum(is_included) AS included_count FROM test WHERE project_id = {project_id} GROUP BY project_id"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['total_count'] - df['included_count']
df

Number of generalization in-/exclusions

In [ ]:
query = f"SELECT project_id, variant, count(*) AS total_count, sum(is_included) AS included_count FROM generalization WHERE project_id = {project_id} GROUP BY project_id, variant"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['total_count'] - df['included_count']
df

Mutation testing results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, status FROM pit_mutation_report WHERE project_id = {project_id}"
df = pd.read_sql_query(query, conn)

df['variant'] = df['variant'].fillna('ORIGINAL')

mutation_status_categories = ['SURVIVED', 'KILLED', 'TIMED_OUT', 'NO_COVERAGE', 'NON_VIABLE', 'MEMORY_ERROR', 'RUN_ERROR']
mutation_status_categories = [c for c in mutation_status_categories if c in df['status'].unique()]
df['status'] = pd.Categorical(df['status'], categories=mutation_status_categories)

df = df.pivot_table(index=['project_id', 'step', 'stage', 'variant'], columns='status', aggfunc='size', fill_value=0, observed=True)
df['TOTAL'] = df.sum(axis=1)
df = df[['TOTAL'] + mutation_status_categories]
df['% killed of covered'] = df['KILLED'] / (df['TOTAL'] - df['NO_COVERAGE'])

df.reset_index(inplace=True)
df

Code coverage results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, sum(instruction_missed), sum(instruction_covered), sum(branch_missed), sum(branch_covered) FROM jacoco_coverage_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant"
df = pd.read_sql_query(query, conn)
df

Runtime requirements per processing stage

In [ ]:
query = f"SELECT project_id, step, stage, sum(runtime) AS runtime FROM task WHERE project_id = {project_id} GROUP BY project_id, step, stage ORDER BY project_id, step"
df = pd.read_sql_query(query, conn)
df

In [ ]:
df['step_stage'] = df['step'].astype(str) + "-" + df['stage'].astype(str)

df.plot(kind='bar', x='step_stage', y='runtime', legend=None, figsize=(12, 6))

plt.xlabel('Processing Stage')
plt.ylabel('Runtime (in seconds)')
plt.title('Runtime per Processing Stage')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

Causes of test failures

In [ ]:
query = f"SELECT project_id, step, stage, variant, failure_type, sum(runtime) FROM junit_test_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant, failure_type"
df = pd.read_sql_query(query, conn)
df['variant'] = df['variant'].fillna('ORIGINAL')
df['failure_type'] = df['failure_type'].fillna('')
df